# Named Entity Recognition for Business Applications

**Course:** Natural Language Processing · Unit 4 — Sequence Labelling and Named Entity Recognition  
**Institution:** Universidad Tecnológica de Bolívar  
**Reference:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3rd ed.), Ch. 8 — Sequence Labelling for Parts of Speech and Named Entities  
**Python compatibility:** 3.10 · 3.11 · 3.12 · 3.13

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Load a spaCy language model and explore its NER pipeline component
2. Extract named entities (ORG, PER, LOC, DATE, MONEY) from financial news
3. Build structured DataFrames from unstructured text using NER
4. Apply NER to email parsing, brand monitoring on social media, and subject-verb-object triplets
5. Use spaCy's `Matcher` to detect custom commercial relationship patterns
6. Visualise entity annotations with `displacy`


---

## 1. Environment Setup

Install dependencies **once** in your terminal:

```bash
pip install spacy
python -m spacy download es_core_news_sm
```


In [ ]:
# pip install spacy  # run once in terminal
# python -m spacy download es_core_news_sm  # run once in terminal
import pandas as pd
import spacy
from spacy.matcher import Matcher
from collections import defaultdict


---

## 2. Load the Language Model

`es_core_news_sm` is spaCy's small Spanish news model. It includes a tokeniser, POS tagger, dependency parser, and NER component trained on news corpora.


In [ ]:
try:
    nlp = spacy.load('es_core_news_sm')
    print(f'Model loaded: {nlp.meta["name"]} v{nlp.meta["version"]}')
    print(f'Pipeline components: {nlp.pipe_names}')
except OSError:
    raise SystemExit(
        'Model not found. Run: python -m spacy download es_core_news_sm'
    )


> **Output interpretation:** The model version confirms which weights are in use. The pipeline components (`['tok2vec', 'morphologizer', 'parser', 'senter', 'ner', ...]`) show all active processing steps — each text passes through them in order. The `ner` component is the one that assigns entity labels.


---

## 3. Business Case 1 — Financial News Entity Extraction

**Scenario:** The business intelligence department needs to automatically extract company names, persons, locations, and monetary figures from financial news headlines to feed a structured database.


In [ ]:
financial_news = [
    'Bancolombia reportó ganancias de 2.3 billones de pesos en el tercer trimestre.',
    'El CEO de Ecopetrol, Ricardo Roa, anunció nuevas inversiones en el Caribe.',
    'Grupo Nutresa adquirió una empresa chilena por 450 millones de dólares.',
    'La Bolsa de Valores de Colombia cerró con una caída del 1.2% este martes.',
    'Avianca reanudará vuelos directos a Madrid a partir del próximo mes.',
]

print('Entities detected in financial news:')
print(f'{"News":60} {"Entity":20} {"Label":10}')
print('-' * 95)
for news in financial_news:
    doc = nlp(news)
    for ent in doc.ents:
        print(f'{news[:58]:60} {ent.text:20} {ent.label_:10}')


> **Output interpretation:** spaCy assigns BILUO tags internally and surfaces the final entity spans. Common labels in this Spanish model: `ORG` (organisation), `PER` (person), `LOC` (location), `MISC` (miscellaneous), `MONEY` (monetary value). Not every entity is correctly classified — the small model may confuse proper nouns and struggle with domain-specific financial terminology.


### 3.1 Build a Structured DataFrame from News Entities

We convert the raw entity annotations into a tabular format suitable for downstream analytics.


In [ ]:
rows = []
for news in financial_news:
    doc = nlp(news)
    for ent in doc.ents:
        rows.append({
            'source_text': news[:60] + '...' if len(news) > 60 else news,
            'entity': ent.text,
            'label': ent.label_,
            'start_char': ent.start_char,
            'end_char': ent.end_char,
        })

df_entities = pd.DataFrame(rows)
print(df_entities.to_string(index=False))
print(f'\nTotal entities extracted: {len(df_entities)}')
print(df_entities['label'].value_counts().to_string())


> **Output interpretation:** The DataFrame structure makes entity data easy to query, filter, and export to a database. The label distribution reveals which entity types appear most in this corpus. A high count of `ORG` entities is expected in financial news. The `start_char`/`end_char` offsets allow the application to highlight spans in the original text (e.g., in a UI or PDF annotation tool).


---

## 4. Business Case 2 — Entity Extraction from Corporate Emails

**Scenario:** The sales team receives hundreds of emails daily. This pipeline automatically extracts contact names, company names, and meeting dates to pre-fill a CRM system.


In [ ]:
corporate_emails = [
    'Buenos días, soy Carlos Martínez de Tecnología XYZ. '
    'Quiero agendar una reunión el próximo lunes 15 de enero.',
    'Hola, María López de Inversiones del Norte necesita hablar '
    'con ustedes sobre el contrato firmado en Bogotá.',
    'El señor Pedro Gómez de Constructora Andina solicita '
    'una cotización para el proyecto en Medellín.',
]

print('Entity extraction from corporate emails:')
for email in corporate_emails:
    doc = nlp(email)
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    print(f'\nEmail: {email[:70]}...')
    print(f'Entities: {entities}')


> **Output interpretation:** Each email yields a list of `(entity_text, label)` tuples. These can be mapped directly to CRM fields: `PER` → contact name, `ORG` → company, `LOC` → city, `DATE` → meeting date. Note that small models trained on news may underperform on conversational email style — for production use, fine-tune on domain-specific annotated emails.


---

## 5. Business Case 3 — Brand Monitoring on Social Media

**Scenario:** The marketing team needs to track which brands, products, and persons are mentioned alongside the company name in social media posts.


In [ ]:
social_posts = [
    'El nuevo iPhone de Apple está destruyendo a Samsung en Colombia. #tech',
    'Ecopetrol y Chevron anunciaron una alianza estratégica para exploración offshore.',
    'Rappi superó a Domicilios.com en descargas durante el último trimestre.',
    'El presidente Gustavo Petro firmó el acuerdo con el Fondo Monetario Internacional.',
]

# Aggregate entity mentions across all posts
brand_mentions: dict = defaultdict(list)
for post in social_posts:
    doc = nlp(post)
    for ent in doc.ents:
        if ent.label_ in ('ORG', 'PER', 'MISC'):
            brand_mentions[ent.label_].append(ent.text)

for label, mentions in brand_mentions.items():
    print(f'[{label}] {mentions}')


> **Output interpretation:** Grouping mentions by label gives the marketing team a quick view of which organisations and persons co-occur with the target brand. Aggregated over thousands of posts, this becomes a brand co-mention graph useful for competitive intelligence.


---

## 6. Advanced Pipeline — spaCy Matcher for Commercial Relationship Patterns

The `Matcher` lets us define token-level rules to capture specific syntactic patterns — for example, acquisition or partnership announcements.


In [ ]:
matcher = Matcher(nlp.vocab)

# Rule: [ENTITY] acquired/bought/merged with [ENTITY]
acquisition_pattern = [
    {'ENT_TYPE': 'ORG'},
    {'LEMMA': {'IN': ['adquirir', 'comprar', 'fusionar', 'aliarse']}},
    {'OP': '?'},
    {'ENT_TYPE': 'ORG'},
]
matcher.add('COMMERCIAL_RELATION', [acquisition_pattern])

business_texts = [
    'Grupo Nutresa adquirió Chocolates Andinos en una operación valorada en 300 millones.',
    'Bancolombia compró el Banco Agrario y ampliará su red rural.',
    'ISA se fusionó con Celsia para crear el mayor holding energético del país.',
]

print('Commercial relationships detected:')
for text in business_texts:
    doc = nlp(text)
    matches = matcher(doc)
    for match_id, start, end in matches:
        span = doc[start:end]
        print(f'  Pattern: {span.text!r} in → {text[:60]}')


> **Output interpretation:** The `Matcher` finds spans matching the defined token pattern regardless of surrounding context. This rule-based approach complements statistical NER: it is fast, transparent, and easy to extend with domain vocabulary. Combine it with NER entity filtering (`ENT_TYPE`) to ensure both sides of the relation are organisations.


---

## 7. Subject–Verb–Object Triplets for Knowledge Extraction

Dependency parsing lets us extract structured (subject, verb, object) triplets from text — the foundation of automated knowledge graph construction.


In [ ]:
def extract_svo_triplets(text: str) -> list:
    """Extract subject-verb-object triplets from text using dependency parsing.

    Args:
        text: Input sentence in Spanish.

    Returns:
        List of (subject, verb, object) string tuples.
    """
    doc = nlp(text)
    triplets = []
    for token in doc:
        if token.dep_ == 'ROOT' and token.pos_ == 'VERB':
            subject = next(
                (t.text for t in token.lefts if t.dep_ in ('nsubj', 'nsubj:pass')),
                None,
            )
            obj = next(
                (t.text for t in token.rights if t.dep_ in ('obj', 'dobj', 'obl')),
                None,
            )
            if subject and obj:
                triplets.append((subject, token.lemma_, obj))
    return triplets


sample_sentences = [
    'Bancolombia anunció resultados positivos en el segundo trimestre.',
    'El gobierno firmó el contrato con Ecopetrol.',
    'Apple lanzó el nuevo iPhone en Colombia.',
]

print('Subject-Verb-Object triplets:')
for sent in sample_sentences:
    triplets = extract_svo_triplets(sent)
    print(f'  Text: {sent}')
    print(f'  Triplets: {triplets}')
    print()


> **Output interpretation:** SVO triplets convert unstructured sentences into (entity, relation, entity) facts that can populate a knowledge graph or be stored in a relational database. The dependency labels `nsubj` (nominal subject), `obj` (direct object), and `obl` (oblique nominal) are language-universal in the Universal Dependencies framework used by spaCy.


---

## 8. Visualisation with displaCy

`displacy` renders entity annotations as highlighted HTML — useful for inspecting model output during development.


In [ ]:
from spacy import displacy

sample = 'Bancolombia y Ecopetrol firmaron un acuerdo en Bogotá el 15 de enero de 2024.'
doc = nlp(sample)

# Render entity annotations in the notebook
displacy.render(doc, style='ent', jupyter=True)


> **Output interpretation:** displaCy colours each entity span by label. This is a quick visual sanity-check of NER quality. In production, use `displacy.serve()` to expose an annotation interface on a local port, or `displacy.render(doc, style='ent', page=True)` to export a standalone HTML file for stakeholder review.


---

## Summary

| Technique | spaCy component | Business use case |
|---|---|---|
| Named entity recognition | `ner` | Extract ORG/PER/LOC/DATE from news and emails |
| Rule-based matching | `Matcher` | Detect acquisition/partnership patterns |
| Dependency parsing | `parser` | Build SVO triplets for knowledge graphs |
| Visualisation | `displacy` | Quality inspection and stakeholder demos |

For higher accuracy on domain-specific text, fine-tune `es_core_news_sm` on annotated business documents using `nlp.update()` with `spacy train`.
